# Tabela `dim_cliente` (silver)

Carrega `customer` do bronze (`nessie.bronze_sakila.customer`), remove
duplicatas, renomeia as colunas para o padrao em portugues e escreve como
`dim_cliente` no schema `silver`. A escrita é
idempotente (`createOrReplace`), então reexecuções recriam a tabela sem
duplicar dados.

In [ ]:
from pyspark.sql import SparkSession

SPARK_APP_NAME = "cria-dim-customer-silver"

from nucleo import cria_spark_session

spark: SparkSession = cria_spark_session(SPARK_APP_NAME)
spark

In [ ]:
TABELA_ORIGEM = "nessie.bronze_sakila.customer"
TABELA_DESTINO = "nessie.silver.dim_cliente"

df = (
    spark.table(TABELA_ORIGEM)
    .dropDuplicates()
    .withColumnRenamed("customer_id", "cliente_id")
    .withColumnRenamed("store_id", "loja_id")
    .withColumnRenamed("first_name", "nome")
    .withColumnRenamed("last_name", "sobrenome")
    .withColumnRenamed("address_id", "endereco_id")
    .withColumnRenamed("active", "ativo")
    .withColumnRenamed("create_date", "data_criacao")
    .withColumnRenamed("last_update", "ultima_atualizacao")
)
df.writeTo(TABELA_DESTINO).createOrReplace()

In [ ]:
n = spark.table(TABELA_DESTINO).count()
print(f"{TABELA_DESTINO}: {n} linhas")